# Four-model comparison and deployment choice
DS483 · Student 4 · Task 5, comparison



In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

frozen = pd.read_csv("splits.csv", keep_default_na=False).query("split == 'test'").set_index("message_id").sort_index()
assert len(frozen) == 797 and frozen.index.is_unique
pred_files = ["student2_predictions.csv", "student3_predictions.csv"]
score_files = ["student2_metrics.csv", "student3_metrics.csv"]
stacked = pd.concat([pd.read_csv(p) for p in pred_files], ignore_index=True)
reported = pd.concat([pd.read_csv(p) for p in score_files], ignore_index=True).set_index("model")
comparison_rows = []
for name, group in stacked.groupby("model", sort=False):
    group = group.set_index("message_id").sort_index()
    assert group.index.equals(frozen.index) and group.actual.equals(frozen.label)
    true = group.actual.eq("spam")
    predicted = group.predicted.eq("spam")
    assert np.array_equal(group.spam_score >= reported.at[name, "cutoff"], predicted)
    tn, fp, fn, tp = confusion_matrix(true, predicted, labels=[False, True]).ravel()
    row = dict(model=name, accuracy=accuracy_score(true,predicted), spam_precision=precision_score(true,predicted),
               spam_recall=recall_score(true,predicted), spam_f1=f1_score(true,predicted),
               macro_f1=f1_score(true,predicted,average="macro"), TN=tn, FP=fp, FN=fn, TP=tp)
    assert all(np.isclose(value, reported.at[name,key]) for key,value in row.items() if key != "model")
    comparison_rows.append(row)
summary = pd.DataFrame(comparison_rows)
assert set(summary.model) == {"Naive Bayes","Logistic Regression","LSTM","BERT-Tiny"}
summary.to_csv("four_model_metrics.csv", index=False)
print(summary.round(4).to_string(index=False))

              model  accuracy  spam_precision  spam_recall  spam_f1  macro_f1  TN  FP  FN  TP
        Naive Bayes    0.9875          0.9619       0.9439   0.9528    0.9728 686   4   6 101
Logistic Regression    0.9812          0.9107       0.9533   0.9315    0.9603 680  10   5 102
               LSTM    0.9787          0.8879       0.9626   0.9238    0.9557 677  13   4 103
          BERT-Tiny    0.9875          0.9450       0.9626   0.9537    0.9732 684   6   4 103


In [2]:
wide = stacked.pivot(index="message_id", columns="model", values="predicted").loc[frozen.index]
wide.insert(0, "actual", frozen.label)
correct = wide.drop(columns="actual").eq(frozen.label, axis=0)
wide["models_correct"] = correct.sum(axis=1)
wide.to_csv("four_model_predictions.csv")
print("All four correct:", int(wide.models_correct.eq(4).sum()))
print("All four wrong:", int(wide.models_correct.eq(0).sum()))
print("BERT correct / NB wrong:", int((correct["BERT-Tiny"] & ~correct["Naive Bayes"]).sum()))
print("NB correct / BERT wrong:", int((~correct["BERT-Tiny"] & correct["Naive Bayes"]).sum()))
for key in ["sms_01291", "sms_03825", "sms_01613"]:
    print(key, "|", frozen.at[key,"text"])
    print(wide.loc[key].to_string(), "\n")

All four correct: 766
All four wrong: 3
BERT correct / NB wrong: 5
NB correct / BERT wrong: 5
sms_01291 | Hey...Great deal...Farm tour 9am to 5pm $95/pax, $50 deposit by 16 May
model
actual                  ham
BERT-Tiny               ham
LSTM                   spam
Logistic Regression    spam
Naive Bayes            spam
models_correct            1 

sms_03825 | Please protect yourself from e-threats. SIB never asks for sensitive information like Passwords,ATM/SMS PIN thru email. Never share your password with anybody.
model
actual                  ham
BERT-Tiny              spam
LSTM                    ham
Logistic Regression     ham
Naive Bayes             ham
models_correct            3 

sms_01613 | 645
model
actual                  ham
BERT-Tiny              spam
LSTM                   spam
Logistic Regression    spam
Naive Bayes            spam
models_correct            0 



In [3]:
timing = pd.read_csv("cpu_timing.csv")
print(timing.round(2).to_string(index=False))

              model  median_ms_batch64
        Naive Bayes               1.20
Logistic Regression               0.97
               LSTM               2.80
          BERT-Tiny              59.49
